# 07 Time Series and Forecasting: From Rolling Averages to ARIMA/SARIMA

The Legionnaires' disease outbreak at Pine and Cypress Nursing Home has entered its second week, and the supervisor throws out two questions:
> 1. "How many more people will get sick next week? How many hospital beds do we need to prepare?"
> 2. "Will **tomorrow** be another peak day? Should we trigger an alert early?"

This notebook answers these two questions with **six models**: rolling mean, Poisson+lag, Negative Binomial+lag, Logistic, ARIMA, and SARIMA.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Build the daily onset-count series ---
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (prevents Chinese labels from showing as tofu boxes) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

# Daily onset counts, filling in dates with no onsets (ensuring continuity)
daily = cases.groupby("symptom_onset_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"Series length: {len(daily)} days | Total cases: {daily.sum()}")
print(daily.head(10))

In [ ]:
# --- Step 2: Epidemic curve + 7-day rolling average ---
rolling_7 = daily.rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#6A9BCC", edgecolor="white", alpha=0.65, label="Daily new cases")
ax.plot(rolling_7.index, rolling_7.values, color="#D97757", linewidth=2,
        label="7-day rolling average")
ax.set_title("Pine and Cypress Nursing Home Legionnaires' Disease Epidemic Curve", fontweight="bold")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")
ax.legend(); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## Part A ── Short-term outbreak forecasting (nursing home data)

The 17 days of outbreak data are enough to train rolling mean / Poisson / NB / Logistic, but **not enough to train ARIMA / SARIMA** (that's Part B's job).

In [ ]:
# --- Step 3: Baseline — Rolling mean forecast ---
# Predict the next day using the average of the previous w days; shift(1) avoids data leakage
print("=== Rolling mean (MAE for different windows) ===")
mae_by_window = {}
for w in [3, 5, 7]:
    pred_w = daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = daily.loc[pred_w.index]
    mae_by_window[w] = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_by_window[w]:.3f}")

mae_rolling = mae_by_window[3]
print(f"\n\u2192 Best: window=3, MAE={mae_rolling:.3f}")

In [ ]:
# --- Step 4: Lagged features — turn yesterday and the day before into features ---
ts = daily.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))       # Day number (trend)
ts["lag_1"] = ts["cases"].shift(1)   # Yesterday's case count
ts["lag_2"] = ts["cases"].shift(2)   # The day before yesterday's case count

# The first two rows have no two-day history to reference → NaN → drop with dropna
ts_model = ts.dropna().reset_index(drop=True)
print(ts_model.head())
print(f"\nUsable rows: {len(ts_model)}")

In [ ]:
# --- Step 5: Poisson regression + lag ---
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Poisson GLM: cases ~ yesterday + day before + day trend
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx",
    data=ts_model,
    family=sm.families.Poisson(),
).fit()

pred_pois = model_pois.predict(ts_model)
mae_pois = mean_absolute_error(ts_model["cases"], pred_pois)
print(f"Poisson + lag:  MAE={mae_pois:.3f},  AIC={model_pois.aic:.2f}")

# Interpret the coefficients: exp(β) = incidence rate ratio (IRR)
coef_table = pd.DataFrame({
    "coef (log scale)": model_pois.params,
    "IRR exp(coef)": np.exp(model_pois.params),
})
print("\n=== Coefficient table ===")
print(coef_table.round(3))

In [ ]:
# --- Step 6: Negative Binomial regression — handling overdispersion ---
# First check the dispersion ratio
disp = ts_model["cases"].var() / ts_model["cases"].mean()
print(f"dispersion = variance / mean = {disp:.2f}")
print("\u2192 > 1.5 is considered overdispersion \u2192 switch to Negative Binomial\n")

# Negative Binomial GLM
model_nb = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx",
    data=ts_model,
    family=sm.families.NegativeBinomial(alpha=1.0),
).fit()

pred_nb = model_nb.predict(ts_model)
mae_nb = mean_absolute_error(ts_model["cases"], pred_nb)
print(f"Negative Binomial + lag:  MAE={mae_nb:.3f},  AIC={model_nb.aic:.2f}")

In [ ]:
# --- Step 7: Logistic regression — Will tomorrow be a peak day? ---
# Use the 75th percentile as the peak day threshold
threshold = ts_model["cases"].quantile(0.75)
ts_model["high_day"] = (ts_model["cases"] > threshold).astype(int)
print(f"Peak day threshold (>75th) = {threshold:.0f} people")
print(f"Peak days: {ts_model['high_day'].sum()} / {len(ts_model)} days\n")

# Use yesterday's and the day before's case counts to predict whether tomorrow exceeds the threshold
model_logit = smf.logit("high_day ~ lag_1 + lag_2", data=ts_model).fit(disp=False)
prob = model_logit.predict(ts_model)
pred_binary = (prob > 0.5).astype(int)
acc = (pred_binary == ts_model["high_day"]).mean()
print(f"Logistic (threshold): accuracy = {acc:.3f}")

print("\n=== Predicted probabilities for the first 5 days ===")
demo = ts_model[["date", "cases", "lag_1", "lag_2", "high_day"]].copy()
demo["P(high)"] = prob.round(3)
print(demo.head())

## Part B ── Long-term surveillance forecasting (synthetic 90-day influenza-like data)

ARIMA / SARIMA need ≥ 30 days (SARIMA needs even more—at least 2 complete cycles). The outbreak data has only 17 days, so forcing it in would be unstable. Here we **synthesize a 90-day daily influenza-like case count** series, including trend + a 7-day weekly cycle + noise.

In [ ]:
# --- Step 8: Synthesize a 90-day surveillance series (trend + 7-day cycle + noise) ---
rng = np.random.default_rng(42)
n_days = 90
dates = pd.date_range("2025-10-01", periods=n_days, freq="D")

trend = np.linspace(3, 7, n_days)                          # Long-term rising trend
seasonal = 3 * np.sin(2 * np.pi * np.arange(n_days) / 7)   # 7-day cycle
noise = rng.normal(0, 1.2, n_days)                          # Random noise
synth_cases = np.maximum(0, (trend + seasonal + noise).round()).astype(int)
synth = pd.Series(synth_cases, index=dates, name="cases")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(synth.index, synth.values, color="#6A9BCC", linewidth=1.5)
ax.set_title("Synthetic influenza-like daily case count (trend + 7-day cycle + noise)", fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Daily case count")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

print(f"Series length: {len(synth)} days | Mean: {synth.mean():.2f} | Variance: {synth.var():.2f}")

In [ ]:
# --- Step 9: ARIMA(1, 1, 1) ---
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Stationarity test
adf_stat, p_value, *_ = adfuller(synth)
print(f"ADF statistic = {adf_stat:.3f}, p-value = {p_value:.3f}")
print("\u2192 p < 0.05 means the series is stationary; otherwise d \u2265 1\n")

# Split train / test: first 83 days for training, last 7 days for testing
train, test = synth.iloc[:-7], synth.iloc[-7:]

model_arima = ARIMA(train, order=(1, 1, 1)).fit()
forecast_arima = model_arima.forecast(steps=7)
mae_arima = mean_absolute_error(test.values, forecast_arima.values)
print(f"ARIMA(1,1,1):  MAE={mae_arima:.3f},  AIC={model_arima.aic:.2f}")

In [ ]:
# --- Step 10: SARIMA(1,1,1)(1,1,1,7) — adding seasonality ---
from statsmodels.tsa.statespace.sarimax import SARIMAX

model_sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
).fit(disp=False)
forecast_sarima = model_sarima.forecast(steps=7)
mae_sarima = mean_absolute_error(test.values, forecast_sarima.values)
print(f"SARIMA(1,1,1)(1,1,1,7):  MAE={mae_sarima:.3f},  AIC={model_sarima.aic:.2f}")

# Visualization: ARIMA vs SARIMA vs actual
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-30:], train.values[-30:], color="#6B6B6B",
        linewidth=1.2, label="Training (last 30 days)")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=5, label="Actual")
ax.plot(test.index, forecast_arima.values, color="#6A9BCC", linewidth=1.8,
        marker="s", markersize=5, linestyle="--", label=f"ARIMA (MAE={mae_arima:.2f})")
ax.plot(test.index, forecast_sarima.values, color="#D97757", linewidth=1.8,
        marker="^", markersize=5, linestyle="--", label=f"SARIMA (MAE={mae_sarima:.2f})")
ax.set_title("ARIMA vs SARIMA 7-day forecast", fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Daily case count")
ax.legend(loc="upper left"); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

### Step 11: Prophet — Meta's "auto-decompose" crystal ball

Prophet **automatically decomposes** the series into three additive building blocks — trend + seasonality + holidays — and **comes with built-in uncertainty intervals**. It only needs two columns, `ds` (date) and `y` (value), and gets you up and running with almost no tuning.

Honest caveat: it's an **additive model**, and on this series with an obvious weekly cycle it performs **about the same as a well-tuned SARIMA** (not magically better). Its real edge is being **easy to use + automatically handling seasonality/holidays/changepoints + providing an interval**.

In [ ]:
# --- Step 11: Prophet — automatically decomposing trend + seasonality + holidays ---
import logging
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)  # silence Stan's noisy logs
from prophet import Prophet

# Prophet only wants two columns: ds (date) + y (value)
pdf = synth.reset_index()
pdf.columns = ["ds", "y"]
p_train = pdf.iloc[:-7]                    # same train/test split as before

m = Prophet(weekly_seasonality=True, yearly_seasonality=False,
            daily_seasonality=False, interval_width=0.9)
m.fit(p_train)                             # auto-detects trend changepoints + fits seasonality

future = m.make_future_dataframe(periods=7)   # extend 7 days into the future
forecast = m.predict(future)
yhat = forecast["yhat"].iloc[-7:].values
mae_prophet = mean_absolute_error(test.values, yhat)
print(f"Prophet:  MAE={mae_prophet:.3f}  (barely any tuning needed, plus built-in uncertainty intervals)")
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(7).to_string(index=False))

In [ ]:
# --- Step 11: Prophet forecast interval visualization ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-30:], train.values[-30:], color="#6B6B6B",
        linewidth=1.2, label="Training (last 30 days)")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=5, label="Actual")
ax.plot(test.index, yhat, color="#788C5D", linewidth=1.8,
        marker="D", markersize=5, linestyle="--", label=f"Prophet (MAE={mae_prophet:.2f})")
ax.fill_between(test.index, forecast["yhat_lower"].iloc[-7:].values,
                forecast["yhat_upper"].iloc[-7:].values,
                color="#788C5D", alpha=0.2, label="90% uncertainty interval")
ax.set_title("Prophet 7-day forecast (with uncertainty interval)", fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Daily case count")
ax.legend(loc="upper left"); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
# --- Step 12: Model showdown ---
comparison = pd.DataFrame([
    {"model": "\u2460 Rolling mean (w=3)",      "dataset": "outbreak",  "MAE": f"{mae_rolling:.3f}",
     "min data": "5 days",  "captures seasonality": "No",     "confidence interval": "No"},
    {"model": "\u2461 Poisson + lag",           "dataset": "outbreak",  "MAE": f"{mae_pois:.3f}",
     "min data": "10 days", "captures seasonality": "Partial", "confidence interval": "Yes"},
    {"model": "\u2462 Negative Binomial + lag", "dataset": "outbreak",  "MAE": f"{mae_nb:.3f}",
     "min data": "10 days", "captures seasonality": "Partial", "confidence interval": "Yes"},
    {"model": "\u2463 Logistic (threshold)",    "dataset": "outbreak",  "MAE": f"— (acc={acc:.2f})",
     "min data": "10 days", "captures seasonality": "No",     "confidence interval": "Yes (probability)"},
    {"model": "\u2464 ARIMA(1,1,1)",            "dataset": "synth 90d", "MAE": f"{mae_arima:.3f}",
     "min data": "30 days", "captures seasonality": "Weak",   "confidence interval": "Yes"},
    {"model": "\u2465 SARIMA(1,1,1)(1,1,1,7)",  "dataset": "synth 90d", "MAE": f"{mae_sarima:.3f}",
     "min data": "60 days", "captures seasonality": "Strong", "confidence interval": "Yes"},
    {"model": "⑦ Prophet",                  "dataset": "synth 90d", "MAE": f"{mae_prophet:.3f}",
     "min data": "~14 days", "captures seasonality": "Strong (auto)", "confidence interval": "Yes"},
])
print(comparison.to_string(index=False))

In [ ]:
# --- Step 13: Onset vs Hospitalization curve (Lag effect) ---
hosp_daily = (
    cases[cases["hospitalization_date"].notna()]
    .groupby("hospitalization_date").size()
)
all_dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
hosp_aligned = hosp_daily.reindex(all_dates, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0, alpha=0.55,
       color="#6A9BCC", edgecolor="white", label="Onset")
ax.bar(hosp_aligned.index, hosp_aligned.values, width=1.0, alpha=0.55,
       color="#D97757", edgecolor="white", label="Hospitalization")
ax.set_title("Onset vs Hospitalization Curve (Lag Effect)", fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Number of People")
ax.legend(); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

lag_days = (hosp_aligned.idxmax() - daily.idxmax()).days
print(f"Onset peak {daily.idxmax().date()} \u2192 hospitalization peak {hosp_aligned.idxmax().date()}")
print(f"Lag = {lag_days} days \u2192 bed and staffing dispatch can start {lag_days} days ahead")

## Summary

| Model | Best-fit situation | Core command |
|------|----------|----------|
| ① Rolling mean | Outbreak just started, very little data | `daily.rolling(w).mean().shift(1)` |
| ② Poisson + lag | Count data + want explanatory variables | `smf.glm(..., family=Poisson())` |
| ③ Negative Binomial | variance ≫ mean (clustering) | `sm.families.NegativeBinomial(alpha=1.0)` |
| ④ Logistic (threshold) | Yes/no alert | `smf.logit('high_day ~ lag_1 + lag_2')` |
| ⑤ ARIMA | ≥ 30 days, no obvious cycle | `ARIMA(y, order=(p,d,q)).fit()` |
| ⑥ SARIMA | ≥ 60 days, has a cycle | `SARIMAX(y, seasonal_order=(P,D,Q,s))` |
| ⑦ Prophet | Has a cycle, want to get started quickly and need an uncertainty interval | `Prophet().fit(df)` |

**Conclusion**: there is no single "best model"—which one to choose depends on the length of your data, whether there's a cycle, and whether you want to predict a **continuous number** or a **yes/no alert**.

In the next chapter (Ch08), we ask "where" it's most severe? → Spatial epidemiology.